# AE2: manejo de archivos CSV, JSON y TXT
Flujo: descarga manual → importar → modificar → guardar nuevo CSV y JSON → tercera clase → resumen TXT.

## 1. URL de origen y rutas relativas

In [1]:
URL_CSV = "https://huggingface.co/datasets/BIT/titanic-dataset/resolve/main/titanic.csv"
URL_JSON = "https://datasets-server.huggingface.co/rows?dataset=BIT%2Ftitanic-dataset&config=default&split=train&offset=0&length=100"

RUTA_CSV = "datos/originales/titanic.csv"
RUTA_JSON = "datos/originales/titanic_filas.json"
RUTA_CSV_NUEVO = "datos/modificados/titanic_modificado.csv"
RUTA_JSON_NUEVO = "datos/modificados/titanic_filas_modificado.json"
RUTA_RESUMEN = "resultados/resumen.txt"

## 2. Importar las clases e inicializar los objetos

In [2]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

csv_m = ManejadorCSV()
json_m = ManejadorJSON()

## 3. CSV: importar, modificar (antes y después) y guardar

In [3]:
csv_m.importar(RUTA_CSV)
print("Encabezados:", csv_m.encabezados)
print("Registros importados:", csv_m.registros_importados)
print("ANTES  fila 0:", csv_m.filas[0])
print("ANTES  total filas:", len(csv_m.filas))

Encabezados: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Registros importados: 891
ANTES  fila 0: ['0', '3', 'male', '22.0', '1', '0', '7.25', 'S', 'Third', 'man', 'True', '', 'Southampton', 'no', 'False']
ANTES  total filas: 891


In [4]:
csv_m.actualizar_valor(0, "age", 23.0)
csv_m.agregar_registro({"survived": 1, "pclass": 2, "sex": "female", "age": 30.0, "fare": 15.5,
                        "embarked": "S", "class": "Second", "who": "woman", "adult_male": False,
                        "embark_town": "Southampton", "alive": "yes", "alone": True,
                        "sibsp": 0, "parch": 0})
csv_m.eliminar_registro(1)
print("DESPUÉS fila 0:", csv_m.filas[0])
print("DESPUÉS última fila:", csv_m.filas[-1])
print("DESPUÉS total filas:", len(csv_m.filas))

DESPUÉS fila 0: ['0', '3', 'male', '23.0', '1', '0', '7.25', 'S', 'Third', 'man', 'True', '', 'Southampton', 'no', 'False']
DESPUÉS última fila: ['1', '2', 'female', '30.0', '0', '0', '15.5', 'S', 'Second', 'woman', 'False', '', 'Southampton', 'yes', 'True']
DESPUÉS total filas: 891


In [5]:
csv_m.guardar(RUTA_CSV_NUEVO)

# verificación: releer el archivo nuevo
verif = ManejadorCSV()
verif.importar(RUTA_CSV_NUEVO)
print("Nuevo CSV:", verif.registros_importados, "registros")
print("age fila 0:", verif.filas[0][verif.encabezados.index("age")])
print("última fila:", verif.filas[-1])

Nuevo CSV: 891 registros
age fila 0: 23.0
última fila: ['1', '2', 'female', '30.0', '0', '0', '15.5', 'S', 'Second', 'woman', 'False', '', 'Southampton', 'yes', 'True']


## 4. JSON: importar, modificar (antes y después) y guardar

In [6]:
json_m.importar(RUTA_JSON)
print("Registros importados:", json_m.registros_importados)
print("ANTES  registro 0:", json_m.datos["rows"][0]["row"])

Registros importados: 100
ANTES  registro 0: {'survived': 0, 'pclass': 3, 'sex': 'male', 'age': 22.0, 'sibsp': 1, 'parch': 0, 'fare': 7.25, 'embarked': 'S', 'class': 'Third', 'who': 'man', 'adult_male': True, 'deck': None, 'embark_town': 'Southampton', 'alive': 'no', 'alone': False}


In [7]:
json_m.actualizar_valor(0, "age", 23.0)
json_m.agregar_registro({"survived": 1, "pclass": 2, "sex": "female", "age": 30.0, "sibsp": 0,
                         "parch": 0, "fare": 15.5, "embarked": "S", "class": "Second",
                         "who": "woman", "adult_male": False, "deck": None,
                         "embark_town": "Southampton", "alive": "yes", "alone": True})
json_m.eliminar_registro(1)
print("DESPUÉS registro 0:", json_m.datos["rows"][0]["row"])
print("DESPUÉS último registro:", json_m.datos["rows"][-1]["row"])
print("DESPUÉS total:", len(json_m.datos["rows"]))

DESPUÉS registro 0: {'survived': 0, 'pclass': 3, 'sex': 'male', 'age': 23.0, 'sibsp': 1, 'parch': 0, 'fare': 7.25, 'embarked': 'S', 'class': 'Third', 'who': 'man', 'adult_male': True, 'deck': None, 'embark_town': 'Southampton', 'alive': 'no', 'alone': False}
DESPUÉS último registro: {'survived': 1, 'pclass': 2, 'sex': 'female', 'age': 30.0, 'sibsp': 0, 'parch': 0, 'fare': 15.5, 'embarked': 'S', 'class': 'Second', 'who': 'woman', 'adult_male': False, 'deck': None, 'embark_town': 'Southampton', 'alive': 'yes', 'alone': True}
DESPUÉS total: 100


In [8]:
json_m.guardar(RUTA_JSON_NUEVO)

# verificación: releer el archivo nuevo
verif_j = ManejadorJSON()
verif_j.importar(RUTA_JSON_NUEVO)
print("Nuevo JSON:", verif_j.registros_importados, "registros")
print("age registro 0:", verif_j.datos["rows"][0]["row"]["age"])

Nuevo JSON: 100 registros
age registro 0: 23.0


## 5. Tercera clase: generar el resumen TXT

In [9]:
generador = GeneradorResumenTXT(URL_CSV, URL_JSON)
contenido = generador.generar(RUTA_RESUMEN, csv_m.informe(), json_m.informe())
print(contenido)

RESUMEN AE2 - MANEJO DE ARCHIVOS CSV, JSON Y TXT
Generado: 2026-10-25 17:41:33
Clase generadora: GeneradorResumenTXT (metodo generar)

== CSV ==
URL de origen: https://huggingface.co/datasets/BIT/titanic-dataset/resolve/main/titanic.csv
Archivo importado: datos/originales/titanic.csv
Clase usada: ManejadorCSV
Metodos usados: importar, actualizar_valor, agregar_registro, eliminar_registro, guardar
Registros importados: 891
Modificaciones realizadas:
  - actualizar_valor: fila 0, columna 'age': 22.0 -> 23.0
  - agregar_registro: nueva fila ['1', '2', 'female', '30.0', '0', '0', '15.5', 'S', 'Second', 'woman', 'False', '', 'Southampton', 'yes', 'True']
  - eliminar_registro: fila 1 eliminada ['1', '1', 'female', '38.0', '1', '0', '71.2833', 'C', 'First', 'woman', 'False', 'C', 'Cherbourg', 'yes', 'False']
Registros finales: 891
Archivo nuevo guardado: datos/modificados/titanic_modificado.csv

== JSON ==
URL de origen: https://datasets-server.huggingface.co/rows?dataset=BIT%2Ftitanic-datas